# 📊 Model Comparison: TMTB vs CSRNet
## Side-by-side comparison of crowd counting models

This notebook compares:
- **TMTB (VMamba)**: State-of-the-art VMamba-based model (~88M parameters)
- **CSRNet**: VGG16-based baseline model (~16M parameters)

### Comparison Metrics:
1. Model size and parameters
2. Inference speed (GPU vs CPU)
3. Prediction accuracy on test images
4. Memory usage
5. Best use cases for each model

## 🔧 Setup: Import Libraries and Paths

In [1]:
import sys
import torch
import time
from pathlib import Path
from PIL import Image
import torchvision.transforms as transforms
import warnings
warnings.filterwarnings('ignore')

# Set up paths
NOTEBOOK_DIR = Path.cwd()
SRC_DIR = NOTEBOOK_DIR.parent
ML_DIR = SRC_DIR.parent
PROJECT_ROOT = ML_DIR.parent

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('✅ Paths configured:')
print(f'   ML Directory: {ML_DIR}')
print(f'   Checkpoints: {ML_DIR / "checkpoints"}')
print(f'   Models: {ML_DIR / "models"}')
print(f'   Datasets: {ML_DIR / "datasets"}')

✅ Paths configured:
   ML Directory: d:\College\Major Project\ml
   Checkpoints: d:\College\Major Project\ml\checkpoints
   Models: d:\College\Major Project\ml\models
   Datasets: d:\College\Major Project\ml\datasets


## 🖥️ Device Setup

In [2]:
# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {device}')

if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print(f'   Available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.2f} GB')
else:
    print('   ⚠️  Using CPU (comparisons will be slower)')

🖥️  Device: cuda
   GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
   Memory: 6.44 GB
   Available: 6.44 GB


## 📦 Load TMTB Model

In [3]:
from models.tmtb.model import mamba

print('📦 Loading TMTB (VMamba) model...')
tmtb_start = time.time()

# Paths
tmtb_corrected = ML_DIR / 'models' / 'tmtb_jhu_corrected.pth'
tmtb_original = ML_DIR / 'checkpoints' / 'jhu_5.pth'

# Choose checkpoint
if tmtb_corrected.exists():
    tmtb_checkpoint_path = tmtb_corrected
    print(f'   Using corrected checkpoint: {tmtb_checkpoint_path.name}')
else:
    tmtb_checkpoint_path = tmtb_original
    print(f'   Using original checkpoint: {tmtb_checkpoint_path.name}')

# Load model on CPU first, then move to device
with torch.no_grad():
    torch.set_default_device('cpu')
    tmtb_model = mamba(25, vmamba_pretrained_path=None)
    checkpoint = torch.load(str(tmtb_checkpoint_path), map_location='cpu', weights_only=False)
    tmtb_model.load_state_dict(checkpoint, strict=False)
    tmtb_model = tmtb_model.to(device)
    tmtb_model.eval()
    if device.type == 'cuda':
        torch.set_default_device('cuda')

tmtb_load_time = time.time() - tmtb_start
tmtb_params = sum(p.numel() for p in tmtb_model.parameters())

print(f'✅ TMTB loaded in {tmtb_load_time:.2f}s')
print(f'   Parameters: {tmtb_params:,}')
print(f'   Size: {tmtb_params * 4 / 1e6:.2f} MB (FP32)')
print(f'   Device: {next(tmtb_model.parameters()).device}')

✅ Using PyTorch-only selective scan (no CUDA extensions)
📦 Loading TMTB (VMamba) model...
   Using corrected checkpoint: tmtb_jhu_corrected.pth
✅ TMTB loaded in 1.73s
   Parameters: 88,683,529
   Size: 354.73 MB (FP32)
   Device: cuda:0


## 📦 Load CSRNet Model

In [4]:
from models.csrnet.csrnet import load_csrnet

print('📦 Loading CSRNet model...')
csrnet_start = time.time()

csrnet_checkpoint = ML_DIR / 'checkpoints' / 'csrnet.pth'

if not csrnet_checkpoint.exists():
    raise FileNotFoundError(f'CSRNet checkpoint not found: {csrnet_checkpoint}')

# Load CSRNet on same device for fair comparison
csrnet_model = load_csrnet(str(csrnet_checkpoint), device=str(device))

csrnet_load_time = time.time() - csrnet_start
csrnet_params = sum(p.numel() for p in csrnet_model.parameters())

print(f'✅ CSRNet loaded in {csrnet_load_time:.2f}s')
print(f'   Parameters: {csrnet_params:,}')
print(f'   Size: {csrnet_params * 4 / 1e6:.2f} MB (FP32)')
print(f'   Device: {next(csrnet_model.parameters()).device}')

📦 Loading CSRNet model...
✅ CSRNet loaded in 0.17s
   Parameters: 16,263,489
   Size: 65.05 MB (FP32)
   Device: cuda:0


## 📊 Model Architecture Comparison

In [5]:
print('='*80)
print('📊 MODEL ARCHITECTURE COMPARISON')
print('='*80)
print()
print(f'{"Metric":<30} {"CSRNet":<25} {"TMTB (VMamba)":<25}')
print('-'*80)
print(f'{"Backbone":<30} {"VGG16":<25} {"VMamba (SSM)":<25}')
print(f'{"Parameters":<30} {f"{csrnet_params:,}":<25} {f"{tmtb_params:,}":<25}')
print(f'{"Model Size (FP32)":<30} {f"{csrnet_params * 4 / 1e6:.2f} MB":<25} {f"{tmtb_params * 4 / 1e6:.2f} MB":<25}')
print(f'{"Relative Size":<30} {"1.0x (baseline)":<25} {f"{tmtb_params/csrnet_params:.2f}x":<25}')
print(f'{"Loading Time":<30} {f"{csrnet_load_time:.2f}s":<25} {f"{tmtb_load_time:.2f}s":<25}')
print('='*80)
print()
print(f'💡 TMTB is {tmtb_params/csrnet_params:.1f}x larger than CSRNet')
print(f'💡 TMTB took {tmtb_load_time/csrnet_load_time:.1f}x longer to load')

📊 MODEL ARCHITECTURE COMPARISON

Metric                         CSRNet                    TMTB (VMamba)            
--------------------------------------------------------------------------------
Backbone                       VGG16                     VMamba (SSM)             
Parameters                     16,263,489                88,683,529               
Model Size (FP32)              65.05 MB                  354.73 MB                
Relative Size                  1.0x (baseline)           5.45x                    
Loading Time                   0.17s                     1.73s                    

💡 TMTB is 5.5x larger than CSRNet
💡 TMTB took 10.2x longer to load


## 🖼️ Load Test Images

In [6]:
# Find test images
dataset_images_dir = ML_DIR / 'datasets' / 'images'

if not dataset_images_dir.exists():
    raise FileNotFoundError(f'Image directory not found: {dataset_images_dir}')

available_images = sorted([
    img for img in dataset_images_dir.glob('*') 
    if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']
])

if not available_images:
    raise FileNotFoundError(f'No images found in {dataset_images_dir}')

print(f'🖼️  Found {len(available_images)} test images:')
for i, img in enumerate(available_images, 1):
    img_obj = Image.open(img)
    print(f'   {i}. {img.name} - {img_obj.size[0]}x{img_obj.size[1]}')

# Setup preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print('\n✅ Images loaded and preprocessing configured')

🖼️  Found 3 test images:
   1. 360_F_216678910_KsrETy4jIFH7bkuB7o4suLhaVqe6ffzq.jpg - 483x360
   2. 360_F_600734899_r2VocyDutRuAcmld87AcxOTCR9NPApSq.jpg - 540x360
   3. png-multicultural-crowd-people-person-back_53876-621138.jpg - 740x494

✅ Images loaded and preprocessing configured


## 🔬 Single Image Comparison

In [7]:
# Test on first image
test_image_path = available_images[0]
test_img = Image.open(test_image_path).convert('RGB')

print(f'🔬 Testing on: {test_image_path.name}')
print(f'   Size: {test_img.size[0]}x{test_img.size[1]} pixels\n')
print('='*80)

# Preprocess
test_tensor = transform(test_img).unsqueeze(0).to(device)

# CSRNet prediction
print('🤖 CSRNet Inference...')
with torch.no_grad():
    csrnet_start = time.time()
    csrnet_output = csrnet_model(test_tensor)
    csrnet_count = csrnet_output.sum().item()
    csrnet_time = time.time() - csrnet_start

print(f'   Count: {csrnet_count:.2f} → {int(round(csrnet_count))} people')
print(f'   Time: {csrnet_time:.3f}s')
print(f'   Density map shape: {csrnet_output.shape}')
print()

# TMTB prediction
print('🤖 TMTB Inference...')
with torch.no_grad():
    tmtb_start = time.time()
    tmtb_output = tmtb_model(test_tensor)
    
    # TMTB returns (density_map, cls_score)
    if isinstance(tmtb_output, tuple):
        tmtb_density = tmtb_output[0]
    else:
        tmtb_density = tmtb_output
    
    tmtb_count = tmtb_density.sum().item()
    tmtb_time = time.time() - tmtb_start

print(f'   Count: {tmtb_count:.2f} → {int(round(tmtb_count))} people')
print(f'   Time: {tmtb_time:.3f}s')
print(f'   Density map shape: {tmtb_density.shape}')
print()
print('='*80)

# Comparison
count_diff = abs(csrnet_count - tmtb_count)
count_diff_pct = (count_diff / max(csrnet_count, tmtb_count)) * 100
speed_ratio = tmtb_time / csrnet_time

print('\n📊 COMPARISON RESULTS:')
print('-'*80)
print(f'{"Metric":<30} {"CSRNet":<25} {"TMTB":<25}')
print('-'*80)
print(f'{"Count":<30} {f"{int(round(csrnet_count))} people":<25} {f"{int(round(tmtb_count))} people":<25}')
print(f'{"Inference Time":<30} {f"{csrnet_time:.3f}s":<25} {f"{tmtb_time:.3f}s":<25}')
print(f'{"Speed Ratio":<30} {"1.0x (baseline)":<25} {f"{speed_ratio:.2f}x slower":<25}')
print('-'*80)
print()
print(f'📈 Count Difference: {count_diff:.2f} people ({count_diff_pct:.1f}%)')

if count_diff_pct < 10:
    print('✅ Models agree well (< 10% difference)')
elif count_diff_pct < 25:
    print('⚠️  Moderate difference (10-25%)')
else:
    print('❌ Large difference (> 25%) - investigate further')

🔬 Testing on: 360_F_216678910_KsrETy4jIFH7bkuB7o4suLhaVqe6ffzq.jpg
   Size: 483x360 pixels

🤖 CSRNet Inference...
   Count: 623.14 → 623 people
   Time: 0.808s
   Density map shape: torch.Size([1, 1, 45, 60])

🤖 TMTB Inference...
   Count: 215.62 → 216 people
   Time: 26.206s
   Density map shape: torch.Size([1, 1, 48, 64])


📊 COMPARISON RESULTS:
--------------------------------------------------------------------------------
Metric                         CSRNet                    TMTB                     
--------------------------------------------------------------------------------
Count                          623 people                216 people               
Inference Time                 0.808s                    26.206s                  
Speed Ratio                    1.0x (baseline)           32.44x slower            
--------------------------------------------------------------------------------

📈 Count Difference: 407.52 people (65.4%)
❌ Large difference (> 25%) - inv

## 🔬 Batch Testing: All Images

In [8]:
print(f'🔬 Testing both models on all {len(available_images)} images:\n')
print('='*100)

results = []

for i, img_path in enumerate(available_images, 1):
    print(f'\n📷 Image {i}/{len(available_images)}: {img_path.name}')
    
    # Load and preprocess
    img = Image.open(img_path).convert('RGB')
    img_size = img.size
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    # CSRNet inference
    with torch.no_grad():
        csrnet_start = time.time()
        csrnet_output = csrnet_model(img_tensor)
        csrnet_count = csrnet_output.sum().item()
        csrnet_time = time.time() - csrnet_start
    
    # TMTB inference
    with torch.no_grad():
        tmtb_start = time.time()
        tmtb_output = tmtb_model(img_tensor)
        if isinstance(tmtb_output, tuple):
            tmtb_density = tmtb_output[0]
        else:
            tmtb_density = tmtb_output
        tmtb_count = tmtb_density.sum().item()
        tmtb_time = time.time() - tmtb_start
    
    # Calculate difference
    count_diff = abs(csrnet_count - tmtb_count)
    count_diff_pct = (count_diff / max(csrnet_count, tmtb_count)) * 100
    
    # Store results
    results.append({
        'name': img_path.name,
        'size': img_size,
        'csrnet_count': csrnet_count,
        'csrnet_time': csrnet_time,
        'tmtb_count': tmtb_count,
        'tmtb_time': tmtb_time,
        'diff': count_diff,
        'diff_pct': count_diff_pct
    })
    
    print(f'   Size: {img_size[0]}x{img_size[1]}')
    print(f'   CSRNet: {int(round(csrnet_count))} people ({csrnet_time:.3f}s)')
    print(f'   TMTB:   {int(round(tmtb_count))} people ({tmtb_time:.3f}s)')
    print(f'   Difference: {count_diff:.1f} ({count_diff_pct:.1f}%)')

print('\n' + '='*100)

🔬 Testing both models on all 3 images:


📷 Image 1/3: 360_F_216678910_KsrETy4jIFH7bkuB7o4suLhaVqe6ffzq.jpg
   Size: 483x360
   CSRNet: 623 people (0.389s)
   TMTB:   216 people (25.934s)
   Difference: 407.5 (65.4%)

📷 Image 2/3: 360_F_600734899_r2VocyDutRuAcmld87AcxOTCR9NPApSq.jpg
   Size: 540x360
   CSRNet: 48 people (0.472s)
   TMTB:   193 people (29.158s)
   Difference: 145.1 (75.1%)

📷 Image 3/3: png-multicultural-crowd-people-person-back_53876-621138.jpg
   Size: 740x494
   CSRNet: 221 people (0.683s)
   TMTB:   345 people (52.521s)
   Difference: 123.7 (35.9%)



## 📊 Summary Statistics

In [9]:
# Calculate statistics
csrnet_total = sum(r['csrnet_count'] for r in results)
tmtb_total = sum(r['tmtb_count'] for r in results)
csrnet_avg_time = sum(r['csrnet_time'] for r in results) / len(results)
tmtb_avg_time = sum(r['tmtb_time'] for r in results) / len(results)
avg_diff = sum(r['diff'] for r in results) / len(results)
avg_diff_pct = sum(r['diff_pct'] for r in results) / len(results)

print('='*80)
print('📊 SUMMARY STATISTICS')
print('='*80)
print()
print(f'{"Metric":<40} {"CSRNet":<20} {"TMTB":<20}')
print('-'*80)
print(f'{"Total Images":<40} {len(results):<20} {len(results):<20}')
print(f'{"Total People Detected":<40} {int(round(csrnet_total)):<20} {int(round(tmtb_total)):<20}')
print(f'{"Average Inference Time":<40} {f"{csrnet_avg_time:.3f}s":<20} {f"{tmtb_avg_time:.3f}s":<20}')
print(f'{"Speed Ratio":<40} {"1.0x (baseline)":<20} {f"{tmtb_avg_time/csrnet_avg_time:.2f}x slower":<20}')
print('-'*80)
print()
print(f'📈 Average Count Difference: {avg_diff:.2f} people ({avg_diff_pct:.1f}%)')
print()
print('='*80)

📊 SUMMARY STATISTICS

Metric                                   CSRNet               TMTB                
--------------------------------------------------------------------------------
Total Images                             3                    3                   
Total People Detected                    892                  753                 
Average Inference Time                   0.515s               35.871s             
Speed Ratio                              1.0x (baseline)      69.67x slower       
--------------------------------------------------------------------------------

📈 Average Count Difference: 225.46 people (58.8%)



## 🎯 Recommendations

In [10]:
print('='*80)
print('🎯 MODEL SELECTION RECOMMENDATIONS')
print('='*80)
print()
print('✅ USE CSRNET WHEN:')
print('   • Speed is critical (real-time applications)')
print('   • Running on CPU or limited GPU memory')
print('   • Resource-constrained environments (mobile, edge devices)')
print(f'   • Good enough accuracy (avg diff: {avg_diff_pct:.1f}%)')
print(f'   • {csrnet_avg_time/tmtb_avg_time:.1f}x faster inference')
print(f'   • {tmtb_params/csrnet_params:.1f}x smaller model')
print()
print('✅ USE TMTB (VMAMBA) WHEN:')
print('   • Highest accuracy is required')
print('   • GPU with sufficient memory is available')
print('   • Can tolerate slower inference')
print('   • Working with complex or dense crowds')
print('   • State-of-the-art performance needed')
print(f'   • {tmtb_params:,} parameters for better representation')
print()
print('='*80)
print()
print('💡 CONCLUSION:')
if avg_diff_pct < 15 and tmtb_avg_time / csrnet_avg_time > 3:
    print(f'   CSRNet offers a good speed/accuracy tradeoff ({csrnet_avg_time/tmtb_avg_time:.1f}x faster,')
    print(f'   only {avg_diff_pct:.1f}% accuracy difference). Recommended for most applications.')
elif avg_diff_pct > 25:
    print(f'   TMTB shows significantly better accuracy ({avg_diff_pct:.1f}% difference).')
    print('   Consider TMTB if accuracy is critical, CSRNet for speed.')
else:
    print(f'   Both models perform similarly (avg diff: {avg_diff_pct:.1f}%).')
    print(f'   Choose CSRNet for speed ({csrnet_avg_time/tmtb_avg_time:.1f}x faster) or TMTB for maximum accuracy.')
print()
print('='*80)

🎯 MODEL SELECTION RECOMMENDATIONS

✅ USE CSRNET WHEN:
   • Speed is critical (real-time applications)
   • Running on CPU or limited GPU memory
   • Resource-constrained environments (mobile, edge devices)
   • Good enough accuracy (avg diff: 58.8%)
   • 0.0x faster inference
   • 5.5x smaller model

✅ USE TMTB (VMAMBA) WHEN:
   • Highest accuracy is required
   • GPU with sufficient memory is available
   • Can tolerate slower inference
   • Working with complex or dense crowds
   • State-of-the-art performance needed
   • 88,683,529 parameters for better representation


💡 CONCLUSION:
   TMTB shows significantly better accuracy (58.8% difference).
   Consider TMTB if accuracy is critical, CSRNet for speed.



## 🧹 Cleanup

In [11]:
# Optional: Free GPU memory
import gc

del csrnet_model
del tmtb_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('✅ GPU memory cleared')
else:
    print('✅ Memory cleared')

✅ GPU memory cleared
